# Notebook 11 — Diagnostik Data: Kenapa Horizon 1 Collapse & Horizon 5 Bias Terbalik?

**Tujuan:** notebook ini TIDAK melatih model apa pun. Murni analisis DATA (OHLCV +
`sentiment_mean`) untuk mencari akar penyebab dua gejala yang sudah teramati:

1. **Horizon 1 (produksi):** Model A (OHLCV) collapse total ke "selalu Up" (recall=1.0,
   TN=FN=0). Model B (FinBERT-LSTM) collapse total ke "selalu Down" (TN=97,FP=0,FN=124,TP=0)
   — arah collapse-nya BERLAWANAN meski data dasarnya (OHLCV) sama, cuma beda 1 fitur
   tambahan (`sentiment_mean`).
2. **Horizon 5 (notebook 10):** kedua model TIDAK collapse (dua kelas terwakili di
   prediksi) tapi under-perform baseline naif "selalu Up" (Acc 38-40% vs baseline ~65%)
   — bias terlalu sering menebak "Down" padahal aktualnya mayoritas "Up".

**Hipotesis yang diuji di notebook ini (satu bagian = satu hipotesis):**
- **H1 — Regime shift:** distribusi Up/Down bergeser signifikan antar periode
  train/val/test (kronologis 70/15/15), sehingga model yang dikalibrasi ke satu rezim
  gagal di rezim lain.
- **H2 — Non-stasioneritas harga & scaling out-of-range:** `FEATURE_COLS`/
  `OHLCV_FEATURE_COLS` memakai level harga MENTAH (Open/High/Low/Close), bukan return.
  `MinMaxScaler` di-fit HANYA di train (anti-leakage, benar secara metodologi) — tapi
  kalau harga terus naik melewati periode train, nilai val/test bisa jatuh JAUH di luar
  rentang [0,1] yang pernah dilihat model saat training (ekstrapolasi/out-of-distribution).
- **H3 — Korelasi fitur-label tidak stabil (sign-flip) antar split:** kalau arah
  hubungan suatu fitur dengan label BERBALIK antara train dan test, model yang belajar
  dari train otomatis "terbalik" saat diuji di test — ini kandidat penjelasan utama utk
  gejala #2 (horizon 5 bias terbalik).
- **H4 — Drift distribusi fitur (train vs val vs test)** diukur dengan uji KS.
- **H5 — Reaktivitas sentimen:** apakah `sentiment_mean` benar-benar leading (memimpin
  harga) atau cuma reaktif (mengikuti harga) — dan apakah ini konsisten di semua split.
- **H6 — Rezim momentum vs mean-reversion:** apakah hubungan antara return hari ini dan
  return N-hari ke depan berubah tanda sepanjang waktu (menjelaskan kenapa horizon 5
  "terbalik" tapi horizon 1 tidak, atau sebaliknya).
- **H7 — Kenapa arah collapse A vs B berlawanan:** apakah ini murni karena baris data yang
  beda (join sentimen membuang hari tanpa coverage berita) atau karena sinyal
  `sentiment_mean` itu sendiri.

**Metodologi:** split & capping REUSE PERSIS `models/lstm/notebook07_common.py::_load_split_capped()`
— split, tanggal, dan outlier-capping di notebook ini adalah split yang SAMA PERSIS yang
dipakai untuk melatih model produksi (horizon 1) dan model horizon 5 (notebook 10), supaya
diagnosisnya relevan langsung ke model yang sudah dilatih (bukan split yang berbeda/tidak
sebanding).

**Status: notebook ini ditulis LENGKAP tapi BELUM DIEKSEKUSI.** Semua cell `outputs=[]`.
Prioritaskan angka/tabel yang dicetak (bukan cuma plot) — supaya bisa dibaca & disintesis
dari teks output setelah dieksekusi.


## Bagian 0 — Setup & Muat 4 Kombinasi Dataset

4 kombinasi: (Model A/OHLCV, Model B/Joined) × (horizon 1, horizon 5). Split & capping
reuse `_load_split_capped()` — TIDAK reinvent logic split.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if (Path.cwd() / "notebooks").exists() is False else Path.cwd()
if (PROJECT_ROOT / "pipeline").exists() is False:
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import MinMaxScaler

from pipeline.feature_engineering.dataset_builder import (
    FEATURE_COLS,
    OHLCV_FEATURE_COLS,
    load_joined,
    load_ohlcv_only,
)
from models.lstm.notebook07_common import load_split_capped

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

FIG_DIR = PROJECT_ROOT / "notebooks"


In [ ]:
COMBOS = [
    ("A_h1", load_ohlcv_only, ["Volume"], 1, OHLCV_FEATURE_COLS),
    ("A_h5", load_ohlcv_only, ["Volume"], 5, OHLCV_FEATURE_COLS),
    ("B_h1", load_joined, ["Volume", "sentiment_mean"], 1, FEATURE_COLS),
    ("B_h5", load_joined, ["Volume", "sentiment_mean"], 5, FEATURE_COLS),
]

DATASETS = {}
print("=== Ringkasan split (baris & tanggal) per kombinasi ===")
for key, load_fn, capping_cols, horizon, feat_cols in COMBOS:
    df_tr, df_vl, df_te, bounds, n_capped = load_split_capped(load_fn, capping_cols, horizon)
    DATASETS[key] = {
        "train": df_tr, "val": df_vl, "test": df_te,
        "capping_bounds": bounds, "n_capped": n_capped,
        "feature_cols": feat_cols, "capping_cols": capping_cols,
        "horizon": horizon,
    }
    for split_name in ["train", "val", "test"]:
        d = DATASETS[key][split_name]
        print(f"[{key}] {split_name:5s} n={len(d):4d} | "
              f"{d['Date'].min().date()} .. {d['Date'].max().date()}")
    print()


## Bagian A (H1) — Ikhtisar Harga & Batas Split

Plot Close price penuh dengan batas train/val/test ditandai, plus ringkasan angka
perubahan harga per split (supaya bisa dibaca dari teks, bukan cuma plot).


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
df_full_b = load_joined()
ax.plot(df_full_b["Date"], df_full_b["Close"], color="black", linewidth=0.8)

colors = {"train": "#4CAF50", "val": "#FF9800", "test": "#F44336"}
for split_name, color in colors.items():
    d = DATASETS["B_h1"][split_name]
    ax.axvspan(d["Date"].min(), d["Date"].max(), color=color, alpha=0.10, label=split_name)

ax.set_title("Close ^GSPC dengan batas split (Model B, horizon=1, kronologis 70/15/15)")
ax.set_ylabel("Close")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "diag11_close_splits.png", dpi=110)
plt.show()

print("=== Perubahan Close per split (Model B, horizon 1) ===")
for split_name in ["train", "val", "test"]:
    d = DATASETS["B_h1"][split_name]
    chg = (d["Close"].iloc[-1] / d["Close"].iloc[0] - 1) * 100
    print(f"{split_name:5s}: {d['Date'].min().date()} -> {d['Date'].max().date()} | "
          f"Close {d['Close'].iloc[0]:8.1f} -> {d['Close'].iloc[-1]:8.1f}  ({chg:+6.1f}%)  "
          f"min={d['Close'].min():.1f} max={d['Close'].max():.1f}")


## Bagian B (H1) — Distribusi Label (Up%) per Split & Sepanjang Waktu

Tabel ringkas Up% per split untuk semua 4 kombinasi, plus rolling Up% (window 60 hari)
sepanjang seluruh timeline supaya terlihat KAPAN pergeseran rezim terjadi (bukan cuma
angka agregat per split).


In [ ]:
print("=== Up% label per split, semua kombinasi ===")
rows = []
for key, d in DATASETS.items():
    row = {"kombinasi": key, "horizon": d["horizon"]}
    for split_name in ["train", "val", "test"]:
        row[split_name] = round(d[split_name]["label"].mean() * 100, 1)
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))


In [ ]:
df_roll = load_joined().copy()
for h in [1, 5]:
    df_roll[f"label_h{h}"] = (df_roll["Close"].shift(-h) > df_roll["Close"]).astype(float)

ROLL_WIN = 60
df_roll["roll_up_h1"] = df_roll["label_h1"].rolling(ROLL_WIN, min_periods=20).mean() * 100
df_roll["roll_up_h5"] = df_roll["label_h5"].rolling(ROLL_WIN, min_periods=20).mean() * 100

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df_roll["Date"], df_roll["roll_up_h1"], label="Rolling 60d Up% (horizon 1)", linewidth=1.2)
ax.plot(df_roll["Date"], df_roll["roll_up_h5"], label="Rolling 60d Up% (horizon 5)", linewidth=1.2)
ax.axhline(50, color="gray", linestyle="--", linewidth=0.8)
for split_name, color in colors.items():
    d = DATASETS["B_h1"][split_name]
    ax.axvspan(d["Date"].min(), d["Date"].max(), color=color, alpha=0.08)
ax.set_ylabel("Up% (rolling 60 hari)")
ax.set_title("Rolling Up% sepanjang waktu -- horizon 1 vs horizon 5")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "diag11_rolling_up_pct.png", dpi=110)
plt.show()

print("=== Statistik rolling Up% per split (horizon 1 & 5) ===")
for h in [1, 5]:
    col = f"roll_up_h{h}"
    print(f"--- horizon {h} ---")
    for split_name in ["train", "val", "test"]:
        d = DATASETS["B_h1"][split_name]
        mask = (df_roll["Date"] >= d["Date"].min()) & (df_roll["Date"] <= d["Date"].max())
        sub = df_roll.loc[mask, col].dropna()
        print(f"  {split_name:5s}: mean={sub.mean():5.1f}%  std={sub.std():5.1f}%  "
              f"min={sub.min():5.1f}%  max={sub.max():5.1f}%")
    print(f"  seluruh timeline: min={df_roll[col].min():.1f}%  max={df_roll[col].max():.1f}%  "
          f"(rentang swing = {df_roll[col].max() - df_roll[col].min():.1f} poin persentase)")


## Bagian C (H2) — Non-Stasioneritas Harga: Cek Rentang Scaling Out-of-Distribution

`FEATURE_COLS`/`OHLCV_FEATURE_COLS` memakai level harga MENTAH (bukan return). `MinMaxScaler`
di-fit HANYA di train (anti-leakage -- ini benar secara metodologi). Tapi kalau harga
trending naik melewati train, nilai val/test scaled bisa jatuh jauh di luar [0,1] --
input yang model TIDAK PERNAH lihat bentuknya saat training. Ini dihitung persis, bukan
diasumsikan.


In [ ]:
print("=== %% nilai scaled di luar [0,1] (scaler fit HANYA di train) ===")
for key, d in DATASETS.items():
    feat_cols = d["feature_cols"]
    scaler = MinMaxScaler()
    scaler.fit(d["train"][feat_cols].values.astype(float))
    print(f"--- {key} ---")
    for split_name in ["val", "test"]:
        raw = d[split_name][feat_cols].values.astype(float)
        scaled = scaler.transform(raw)
        below0 = (scaled < 0).mean(axis=0) * 100
        above1 = (scaled > 1).mean(axis=0) * 100
        for i, col in enumerate(feat_cols):
            mn, mx = scaled[:, i].min(), scaled[:, i].max()
            flag = " <-- OUT OF RANGE" if (below0[i] > 0 or above1[i] > 0) else ""
            print(f"  [{split_name:5s}] {col:16s} | %<0={below0[i]:5.1f}% %>1={above1[i]:5.1f}% "
                  f"| scaled range=[{mn:6.2f}, {mx:6.2f}]{flag}")
    print()


In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
scaler_close = MinMaxScaler()
train_b = DATASETS["B_h1"]["train"]
scaler_close.fit(train_b[["Close"]].values.astype(float))

full_scaled_close = scaler_close.transform(df_full_b[["Close"]].values.astype(float)).ravel()
ax.plot(df_full_b["Date"], full_scaled_close, color="black", linewidth=0.8)
ax.axhline(0, color="green", linestyle="--", linewidth=1, label="batas bawah train (0)")
ax.axhline(1, color="red", linestyle="--", linewidth=1, label="batas atas train (1)")
for split_name, color in colors.items():
    d = DATASETS["B_h1"][split_name]
    ax.axvspan(d["Date"].min(), d["Date"].max(), color=color, alpha=0.08)
ax.set_title("Close discale dengan MinMaxScaler fit-train-only -- area di luar [0,1] = out-of-distribution")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "diag11_close_scaled_range.png", dpi=110)
plt.show()


## Bagian D (H3) — Korelasi Fitur vs Label per Split: Deteksi Sign-Flip

Kalau arah korelasi suatu fitur dengan label BERBALIK TANDA antara train dan test, model
yang belajar pola dari train otomatis "terbalik" saat dites di test -- kandidat penjelasan
utama untuk horizon 5 yang bias ke arah salah.


In [ ]:
for key in ["A_h1", "B_h1", "A_h5", "B_h5"]:
    d = DATASETS[key]
    feat_cols = d["feature_cols"]
    print(f"=== Korelasi point-biserial (fitur vs label) -- {key} (horizon={d['horizon']}) ===")
    corr_rows = []
    for split_name in ["train", "val", "test"]:
        split_df = d[split_name]
        row = {"split": split_name}
        for col in feat_cols:
            r, p = stats.pointbiserialr(split_df["label"], split_df[col])
            row[col] = round(r, 3)
        corr_rows.append(row)
    corr_df = pd.DataFrame(corr_rows).set_index("split")
    print(corr_df)

    for col in feat_cols:
        signs = np.sign(corr_df[col].values)
        nonzero_signs = set(s for s in signs if s != 0)
        if len(nonzero_signs) > 1:
            print(f"  !! SIGN-FLIP pada '{col}': train={corr_df.loc['train', col]:+.3f} "
                  f"val={corr_df.loc['val', col]:+.3f} test={corr_df.loc['test', col]:+.3f}")
    print()


## Bagian E (H4) — Drift Distribusi Fitur (Train vs Val vs Test)

Uji Kolmogorov-Smirnov train-vs-test per fitur (kuantitatif) + histogram overlay
(visual) untuk kombinasi B (superset fitur, mencakup OHLCV + sentimen).


In [ ]:
print("=== KS-test train vs test per fitur (drift distribusi) ===")
for key in ["A_h1", "B_h1"]:
    d = DATASETS[key]
    print(f"--- {key} ---")
    for col in d["feature_cols"]:
        stat_ks, p = stats.ks_2samp(d["train"][col], d["test"][col])
        flag = " <-- DRIFT SIGNIFIKAN (p<0.01)" if p < 0.01 else ""
        print(f"  {col:16s} | KS={stat_ks:.3f}  p={p:.4f}{flag}")
    print()


In [ ]:
feat_cols_b = DATASETS["B_h1"]["feature_cols"]
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flat, feat_cols_b):
    for split_name, color in colors.items():
        d = DATASETS["B_h1"][split_name]
        ax.hist(d[col], bins=30, alpha=0.5, label=split_name, color=color, density=True)
    ax.set_title(col)
    ax.legend(fontsize=8)
plt.suptitle("Distribusi fitur per split (Model B, horizon 1) -- overlap kecil = drift besar")
plt.tight_layout()
plt.savefig(FIG_DIR / "diag11_feature_drift_hist.png", dpi=110)
plt.show()


## Bagian F (H5) — Reaktivitas Sentimen: Lead-Lag Cross-Correlation

Korelasi `sentiment_mean(t)` vs return `Close` di lag 0..10 hari ke depan, dihitung utk
seluruh data DAN per split -- untuk melihat apakah sentimen benar-benar memimpin harga
atau cuma mengikuti (reaktif), dan apakah polanya stabil antar periode.


In [ ]:
df_lag = load_joined().copy()
df_lag["ret_1"] = df_lag["Close"].pct_change(1)

max_lag = 10
print("=== Korelasi sentiment_mean(t) vs return Close(t+k), k=0..10 (seluruh data) ===")
full_rows = []
for k in range(0, max_lag + 1):
    fwd = df_lag["Close"].pct_change(1).shift(-k)
    valid = df_lag["sentiment_mean"].notna() & fwd.notna()
    r, p = stats.pearsonr(df_lag.loc[valid, "sentiment_mean"], fwd[valid])
    full_rows.append({"lag_hari": k, "pearson_r": round(r, 4), "p_value": round(p, 4)})
print(pd.DataFrame(full_rows).to_string(index=False))

print()
print("=== Korelasi sentiment_mean(t) vs return Close(t+k) PER SPLIT (Model B, horizon 1) ===")
for split_name in ["train", "val", "test"]:
    d = DATASETS["B_h1"][split_name].copy().reset_index(drop=True)
    print(f"--- {split_name} ---")
    rows = []
    for k in range(0, max_lag + 1):
        fwd = d["Close"].pct_change(1).shift(-k)
        valid = d["sentiment_mean"].notna() & fwd.notna()
        if valid.sum() < 10:
            continue
        r, p = stats.pearsonr(d.loc[valid, "sentiment_mean"], fwd[valid])
        rows.append({"lag_hari": k, "pearson_r": round(r, 4), "p_value": round(p, 4)})
    print(pd.DataFrame(rows).to_string(index=False))


## Bagian G (H6) — Rezim Momentum vs Mean-Reversion Sepanjang Waktu

Rolling correlation antara return hari ini (`ret_1`) dan return N-hari ke depan
(`fwd_ret_5`). Korelasi positif = momentum (tren berlanjut), korelasi negatif =
mean-reversion (tren berbalik). Kalau tanda korelasi ini BERUBAH antar periode
train/val/test, itu bukti langsung kenapa model horizon-5 bisa "terbalik".


In [ ]:
df_mr = load_joined().copy()
df_mr["ret_1"] = df_mr["Close"].pct_change(1)
df_mr["fwd_ret_5"] = df_mr["Close"].pct_change(5).shift(-5)

ROLL_MR = 120
df_mr["roll_mr_corr"] = df_mr["ret_1"].rolling(ROLL_MR).corr(df_mr["fwd_ret_5"])

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df_mr["Date"], df_mr["roll_mr_corr"], color="purple", linewidth=1)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
for split_name, color in colors.items():
    d = DATASETS["B_h1"][split_name]
    ax.axvspan(d["Date"].min(), d["Date"].max(), color=color, alpha=0.08)
ax.set_title(f"Rolling {ROLL_MR}d correlation: return hari-ini vs return 5-hari ke depan "
             "(positif=momentum, negatif=mean-reversion)")
ax.set_ylabel("Korelasi")
plt.tight_layout()
plt.savefig(FIG_DIR / "diag11_momentum_reversion.png", dpi=110)
plt.show()

print("=== Rata-rata korelasi momentum/mean-reversion per split ===")
for split_name in ["train", "val", "test"]:
    d = DATASETS["B_h1"][split_name]
    mask = (df_mr["Date"] >= d["Date"].min()) & (df_mr["Date"] <= d["Date"].max())
    sub = df_mr.loc[mask, "roll_mr_corr"].dropna()
    if len(sub) == 0:
        print(f"  {split_name:5s}: (data tidak cukup utk rolling window)")
        continue
    regime = "MOMENTUM (+)" if sub.mean() > 0.05 else ("MEAN-REVERSION (-)" if sub.mean() < -0.05 else "netral/campur")
    print(f"  {split_name:5s}: mean_corr={sub.mean():+.3f}  std={sub.std():.3f}  -> rezim dominan: {regime}")


## Bagian H (kontekstualisasi) — Baseline Sederhana per Split

Always-Majority dan Persistence (label hari ini == label kemarin), untuk keempat
kombinasi -- supaya angka model (dari notebook 07/10) punya pembanding non-ML yang jelas.


In [ ]:
print("=== Baseline sederhana per split, semua kombinasi ===")
for key, d in DATASETS.items():
    print(f"--- {key} (horizon={d['horizon']}) ---")
    for split_name in ["train", "val", "test"]:
        s = d[split_name]
        maj_acc = max(s["label"].mean(), 1 - s["label"].mean())
        persist_acc = (s["label"] == s["label"].shift(1)).mean()
        print(f"  {split_name:5s} n={len(s):4d} | Always-Majority={maj_acc:.3f} | "
              f"Persistence(label[t]==label[t-1])={persist_acc:.3f}")
    print()


## Bagian I (H7) — Kenapa Arah Collapse Model A vs Model B Berlawanan?

Model A (OHLCV) collapse ke "selalu Up", Model B (+`sentiment_mean`) collapse ke
"selalu Down" -- padahal basis datanya nyaris sama. Cek dua kandidat penyebab: (1) baris
data yang beda (inner join sentimen membuang hari tanpa coverage berita), (2) korelasi
`sentiment_mean` itu sendiri dengan label per split.


In [ ]:
dates_a = set(DATASETS["A_h1"]["train"]["Date"]) | set(DATASETS["A_h1"]["val"]["Date"]) | set(DATASETS["A_h1"]["test"]["Date"])
dates_b = set(DATASETS["B_h1"]["train"]["Date"]) | set(DATASETS["B_h1"]["val"]["Date"]) | set(DATASETS["B_h1"]["test"]["Date"])
only_in_a = dates_a - dates_b

print(f"Total hari Model A: {len(dates_a)} | Model B: {len(dates_b)}")
print(f"Hari yang ADA di Model A tapi HILANG di Model B (tanpa coverage sentimen): {len(only_in_a)} "
      f"({len(only_in_a) / len(dates_a) * 100:.1f}% dari total)")

print()
print("=== Korelasi sentiment_mean vs label per split (dari Bagian D, ditarik ulang utk fokus) ===")
d_b = DATASETS["B_h1"]
for split_name in ["train", "val", "test"]:
    split_df = d_b[split_name]
    r, p = stats.pointbiserialr(split_df["label"], split_df["sentiment_mean"])
    print(f"  {split_name:5s}: r={r:+.3f}  p={p:.4f}  "
          f"n_capped(sentiment_mean)={d_b['n_capped'][split_name].get('sentiment_mean', 0)}")

print()
print("=== Bounds capping IQR sentiment_mean (dihitung dari train) ===")
print(d_b["capping_bounds"].get("sentiment_mean"))


## Bagian J — Ringkasan Angka Kunci (untuk sintesis)

Satu cell konsolidasi angka-angka paling menentukan dari seluruh bagian di atas, supaya
gampang dibaca sekali lihat tanpa scroll ke setiap bagian.


In [ ]:
print("=" * 70)
print("RINGKASAN ANGKA KUNCI")
print("=" * 70)

print("\n[H1] Up% per split (B, horizon 1 vs 5):")
for h in [1, 5]:
    key = f"B_h{h}"
    d = DATASETS[key]
    vals = {s: round(d[s]["label"].mean() * 100, 1) for s in ["train", "val", "test"]}
    print(f"  horizon {h}: {vals}")

print("\n[H2] Fitur dengan nilai test scaled di luar [0,1] (Model B, horizon 1):")
scaler_chk = MinMaxScaler()
scaler_chk.fit(DATASETS["B_h1"]["train"][FEATURE_COLS].values.astype(float))
test_scaled_chk = scaler_chk.transform(DATASETS["B_h1"]["test"][FEATURE_COLS].values.astype(float))
for i, col in enumerate(FEATURE_COLS):
    pct_out = ((test_scaled_chk[:, i] < 0) | (test_scaled_chk[:, i] > 1)).mean() * 100
    if pct_out > 0:
        print(f"  {col}: {pct_out:.1f}% baris test di luar [0,1]")

print("\n[H3] Fitur dengan sign-flip korelasi (train vs test), semua kombinasi:")
for key in ["A_h1", "B_h1", "A_h5", "B_h5"]:
    d = DATASETS[key]
    for col in d["feature_cols"]:
        r_tr, _ = stats.pointbiserialr(d["train"]["label"], d["train"][col])
        r_te, _ = stats.pointbiserialr(d["test"]["label"], d["test"][col])
        if np.sign(r_tr) != np.sign(r_te) and r_tr != 0 and r_te != 0:
            print(f"  [{key}] {col}: train={r_tr:+.3f} -> test={r_te:+.3f}")

print("\n[H6] Rezim momentum/mean-reversion rata-rata per split:")
for split_name in ["train", "val", "test"]:
    d = DATASETS["B_h1"][split_name]
    mask = (df_mr["Date"] >= d["Date"].min()) & (df_mr["Date"] <= d["Date"].max())
    sub = df_mr.loc[mask, "roll_mr_corr"].dropna()
    if len(sub):
        print(f"  {split_name:5s}: {sub.mean():+.3f}")

print("\n[H7] Hari hilang krn join sentimen: ", len(only_in_a), f"({len(only_in_a)/len(dates_a)*100:.1f}%)")


## Catatan Penutup

Notebook ini murni **diagnostik data** -- tidak ada model yang dilatih ulang di sini.
Setelah dieksekusi, kirim kembali hasilnya (output tiap cell, terutama Bagian J) untuk
disintesis jadi kesimpulan tertulis tentang akar penyebab collapse (horizon 1) dan bias
terbalik (horizon 5), yang bisa dipakai sebagai dasar keputusan horizon mana yang dipakai
di skripsi -- atau sebagai temuan diagnostik tersendiri untuk BAB V.
